<div align="center">
  <h1>Monitoring Forest Disturbance and Recovery Dynamics in the UNESCO Greater Blue Mountains Area</h1>
  <h3><i>A Multi-temporal NDVI Analysis of the 2019/2020 'Black Summer' Fires</i></h3>
  <br>
  <p><b>Author:</b> Łukasz Słowik &nbsp; | &nbsp; <b>Date:</b> June 2026</p>
  <p><i>Final Assessment: Forest Monitoring with Sentinel-2 & GEE</i></p>
  <hr>
</div>

# 1. Google Cloud Configuration

In [ ]:

# @markdown **Note:** `PROJECT_ID` is the **only variable** that requires user modification
# @markdown to execute the entire analytical pipeline.
# @markdown


PROJECT_ID = "forest-ai-course-2026" #@param {type:"string"}

# 2. Introduction

## 2.1. Project Structure and Workflow

This assessment is organized into a modular, reproducible analytical pipeline. To ensure clarity and maintainability, the notebook follows a strict separation between global configuration, function definitions, and final execution. The project is divided into the following chapters:

*   **Chapter 1: Google Cloud Configuration** – The technical entry point where the user specifies the `PROJECT_ID` required for Earth Engine authentication and resource management.
*   **Chapter 2: Introduction** – Provides the ecological context of the Greater Blue Mountains, defines the research problem (Eucalyptus resilience vs. Rainforest vulnerability), outlines monitoring objectives, and includes a methodology disclosure regarding AI-assisted development.
*   **Chapter 3: Python Environment Setup** – Handles the installation of necessary dependencies (Google Earth Engine API, GeoPandas, geemap) and the import of specialized geospatial libraries.
*   **Chapter 4: Project Constants and Data Assets** – A centralized configuration section for defining Earth Engine Asset IDs, temporal analysis windows (Baseline, Impact, Recovery), and technical parameters such as NDVI thresholds and spatial scales.
*   **Chapter 5: Analytical Methods (Function Definitions)** – Contains the core "Engine" of the project. It defines specialized Python functions for data acquisition (WDPA), raster pre-processing (Sentinel-2), spectral index calculation (NDVI), topographic morphology (TPI), and professional visualization tools.
*   **Chapter 6: Execution and Ecological Interpretation** – The final operational phase where the pipeline is executed. It is organized into two sequential steps:
    1.  **Data Generation:** Server-side computations are performed first to create output datasets (composites, health classifications, and topographic statistics).
    2.  **Visual Reporting and Observations:** The generated data is rendered through interactive dashboards, static figures, and analytical tables, accompanied by a detailed interpretation of the findings, the "recovery paradox," and study limitations.

## 2.2. Ecological Significance



The Greater Blue Mountains World Heritage Area (UNESCO) represents a unique landscape of extreme biodiversity. It is characterized by two contrasting botanical worlds:

- Sclerophyll Forests (Eucalyptus): Pyrophytic species adapted to survive and even thrive on periodic fires. They utilize mechanisms such as epicormic sprouting (buds protected under thick bark) for rapid post-fire regeneration.

- Temperate Rainforests and Refugia: Ancient Gondwana-era relics, including the critically endangered Wollemi Pine (Wollemia nobilis), which seek shelter in deep, moist sandstone canyons. These ecosystems are highly sensitive to fire and rely on topographic protection to survive.

## 2.3. The 'Black Summer' Crisis (2019/20)

During the catastrophic 2019/20 fire season, an unprecedented multi-year drought allowed high-intensity fires to breach natural moisture barriers. For the first time in recorded history, fire penetrated deep into the moist canyon floors, threatening the survival of fire-sensitive refugia.

Further Reading:
- https://whc.unesco.org/en/list/917/
- https://en.wikipedia.org/wiki/2019%E2%80%9320_Australian_bushfire_season

## 2.4. Monitoring Objectives

This project utilizes Sentinel-2 Multi-Spectral data and manually calculated NDVI to:

- Quantify the regional biomass loss across three National Parks: Blue Mountains, Wollemi, and Yengo.

- Differentiate the impact between exposed ridges (fire-prone) and deep canyons (refugia) using topographic analysis (TPI).

- Assess the 10-year vegetation recovery trajectory (2016–2026) to identify areas of successful resilience versus permanent degradation.

## 2.5. AI Collaboration and Methodology Disclosure

While I am a Python user and possess the technical skills to implement this workflow independently, this project was developed through an intensive, continuous collaboration with the Gemini AI model. The AI served as a comprehensive analytical partner throughout the entire process. Specifically, Gemini was utilized for:

  - Documentation: Generating technical docstrings, refining code comments, and ensuring consistent English terminology.

  - Quality Control: Identifying and correcting typographical errors and optimizing function structures.

  - Translation: Translating complex ecological interpretations and technical descriptions from Polish to English.

  - Conceptual Consulting: Brainstorming the topographic analysis (TPI) and validating the "recovery paradox" logic.
  The final notebook is the result of a singular, deep analytical dialogue, where the AI assisted in articulating programming results into a structured Earth Observation report.

# 3. Python Environment and Dependencies Setup

## 3.1 Package Installation

In [ ]:
!pip install -q earthengine-api
!pip install geopandas

## 3.2 Library Imports

In [ ]:
import ee
import geemap
import geopandas as gpd
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import numpy as np

# 4. Project Constants and Data Assets

## 4.1. Data Sources Documentation

The following datasets are pulled directly from the Google Earth Engine Public Catalog:

| Asset Name | Official Data Catalog Link |
| :--- | :--- |
| **WDPA** | [WCMC/WDPA/current/polygons](https://developers.google.com/earth-engine/datasets/catalog/WCMC_WDPA_current_polygons) |
| **Sentinel-2** | [COPERNICUS/S2_HARMONIZED](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_HARMONIZED) |
| **NASADEM** | [NASA/NASADEM_HGT/001](https://developers.google.com/earth-engine/datasets/catalog/NASA_NASADEM_HGT_001) |
| **MODIS Fire** | [MODIS/061/MCD64A1](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MCD64A1) |

## 4.2. Project Constants Definition

In [ ]:
# ==============================================================================
# 1. EARTH ENGINE ASSET PATHS (DATA SOURCES)
# ==============================================================================
WDPA_PATH = "WCMC/WDPA/current/polygons"   # World Database on Protected Areas
S2_PATH   = "COPERNICUS/S2_HARMONIZED"     # Sentinel-2 L1C (Top-of-Atmosphere)
DEM_PATH  = "NASA/NASADEM_HGT/001"         # NASA Digital Elevation Model
FIRE_PATH = "MODIS/061/MCD64A1"            # MODIS Global Burned Area product

# ==============================================================================
# 2. STUDY AREA CONFIGURATION (GREATER BLUE MOUNTAINS)
# ==============================================================================
PARK_NAMES  = ['Blue Mountains', 'Wollemi', 'Yengo']
COUNTRY_ISO = 'AUS'  # ISO3 country code for Australia

# ==============================================================================
# 3. TEMPORAL ANALYSIS WINDOWS (ECOLOGICAL PHASES)
# ==============================================================================
# PHASE 1: Baseline (Pre-fire reference state from a stable hydrological year)
PRE_FIRE_START, PRE_FIRE_END = '2016-12-01', '2017-03-31'

# PHASE 2: Fire Event (Peak impact and immediate burn scar observation)
FIRE_EVENT_START, FIRE_EVENT_END = '2020-01-01', '2020-03-31'

# PHASE 3: Recovery (Current vegetation regrowth status)
# assumed date, to maintain +/- equal comparative periods
POST_FIRE_START, POST_FIRE_END = '2025-12-01', '2026-03-31'

# ==============================================================================
# 4. ECOLOGICAL & ANALYSIS THRESHOLDS
# ==============================================================================
THRESHOLD_HEALTHY  = 0.6    # Minimum NDVI for dense, healthy canopy
THRESHOLD_DEGRADED = 0.3    # Threshold for severe burn or non-forest surface

# ==============================================================================
# 5. TECHNICAL COMPUTATION PARAMETERS
# ==============================================================================
MAX_CLOUD_PERCENTAGE = 30   # Max allowable cloud cover per scene metadata
SCALE_STATS = 100           # Spatial resolution (m) for regional area calculations
SCALE_PLOT  = 200           # Spatial resolution (m) for Matplotlib/NumPy exports
MAX_PIXELS  = 1e13          # Pixel limit for complex Earth Engine reductions

print(f"Configuration successfully loaded for project: {PROJECT_ID}")
print("All study-specific parameters are locked for the Blue Mountains analysis.")

# 5. Analytical Methods and Function Definitions

## 5.1. GEE connection

In [ ]:
def authenticate_project(project_id: str) -> None:
  """
  Authenticates and initializes the Google Earth Engine (GEE) Python API.

  This function handles the OAuth2 authentication flow and links the session
  to a specific Google Cloud Project. It is a mandatory setup step required
  to access GEE's cloud-based geospatial data catalog and server-side
  processing capabilities.

  Args:
      project_id (str): The unique identifier of the Google Cloud Project
                        registered for Earth Engine access.
  """
  ee.Authenticate()
  ee.Initialize(project=project_id)
  print("Earth Engine initialised successfully.")

## 5.2. Core Processing Functions

### 5.2.1. Vector Acquisition and Metadata

In [ ]:
def fetch_study_area(
    park_names: list[str],
    iso_code: str,
    collection_id: str
) -> tuple[ee.Geometry, ee.FeatureCollection]:
    """
    DATA LOGIC: Fetches protected area boundaries from WDPA.
    Returns: (study_area_geom, study_area_features)
    """
    wdpa_database = ee.FeatureCollection(collection_id)
    study_area_features = wdpa_database.filter(ee.Filter.And(
        ee.Filter.inList('NAME', park_names),
        ee.Filter.eq('ISO3', iso_code)
    ))
    study_area_geom = study_area_features.geometry()
    return study_area_geom, study_area_features


def analyze_vector_metadata(study_area_features: ee.FeatureCollection) -> gpd.GeoDataFrame:
    """
    Converts the Earth Engine study area features to a local GeoPandas GeoDataFrame
    for attribute inspection and metadata reporting.

    Args:
        study_area_features (ee.FeatureCollection): The Earth Engine collection of selected parks.

    Returns:
        gpd.GeoDataFrame: A local GeoDataFrame containing park attributes and geometries.
    """
    # Convert Earth Engine FeatureCollection to a local GeoPandas GeoDataFrame
    # This brings the data from GEE servers to the local Python environment
    gdf = geemap.ee_to_gdf(study_area_features)

    # Relevant attributes documentation:
    # 'NAME'       - Official name of the protected area.
    # 'DESIG_ENG'  - Designation type in English (e.g., National Park).
    # 'IUCN_CAT'   - IUCN (International Union for Conservation of Nature) Management Category.
    # 'REP_AREA'   - Reported surface area in square kilometers (km²).
    # 'STATUS_YR'  - The year the area was officially established.

    # Rename the area column to include the unit [km²] for better clarity in reports
    if 'REP_AREA' in gdf.columns:
        gdf = gdf.rename(columns={'REP_AREA': 'REP_AREA [km²]'})

    # Define the list of columns to display in the summary table
    useful_cols = ['NAME', 'DESIG_ENG', 'IUCN_CAT', 'REP_AREA [km²]', 'STATUS_YR']
    existing_cols = [c for c in useful_cols if c in gdf.columns]

    print("Study Area Metadata Table:")
    # Display the filtered DataFrame with selected attributes
    display(gdf[existing_cols])

    return gdf

### 5.2.2. Raster Pre-processing and Spectral Indices

In [ ]:
def generate_fire_analysis_composites(
    study_area_geom: ee.Geometry,
    s2_collection_id: str,
    cloud_threshold: int,
    pre_start: str,
    pre_end: str,
    fire_start: str,
    fire_end: str,
    rec_start: str,
    rec_end: str
) -> tuple[ee.Image, ee.Image, ee.Image]:
    """
    Processes Sentinel-2 imagery to create cloud-masked median composites for three
    ecological phases: Pre-fire, Fire Event, and Post-fire Recovery.

    Args:
        study_area_geom (ee.Geometry): The geometry used for spatial filtering and clipping.
        s2_collection_id (str): The GEE asset ID for Sentinel-2 (e.g., S2_HARMONIZED).
        cloud_threshold (int): Maximum allowable cloud cover percentage per image.
        pre_start, pre_end (str): Date range for the baseline period.
        fire_start, fire_end (str): Date range for the fire impact period.
        rec_start, rec_end (str): Date range for the vegetation recovery period.

    Returns:
        tuple[ee.Image, ee.Image, ee.Image]: (img_pre, img_fire, img_rec) as processed ee.Image objects.
    """

    # 1. Initialize the collection and apply initial spatial and metadata filters
    s2_col = ee.ImageCollection(s2_collection_id) \
        .filterBounds(study_area_geom) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_threshold))

    def mask_s2_toa(image):
        """Internal helper to mask clouds and scale reflectance values for TOA imagery."""
        qa = image.select('QA60')
        # Bitmask for clouds (10) and cirrus (11)
        mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
        return image.updateMask(mask).divide(10000)

    # 2. Generate cloud-masked median composites for each phase
    # Median reduction is used to synthesize a single cloud-free image from the collection

    # Phase 1: Pre-fire Baseline
    img_pre = s2_col.filterDate(pre_start, pre_end) \
                    .map(mask_s2_toa).median().clip(study_area_geom)

    # Phase 2: Fire Event / Impact Scar
    img_fire = s2_col.filterDate(fire_start, fire_end) \
                     .map(mask_s2_toa).median().clip(study_area_geom)

    # Phase 3: Post-fire Recovery / Regeneration
    img_rec = s2_col.filterDate(rec_start, rec_end) \
                    .map(mask_s2_toa).median().clip(study_area_geom)

    return img_pre, img_fire, img_rec


def calculate_ndvi(image: ee.Image) -> ee.Image:
    """
    Calculates the Normalized Difference Vegetation Index (NDVI) manually
    from Sentinel-2 spectral bands.

    This function satisfies the assessment requirement for a custom Python
    function. It implements the standard vegetation index formula using
    Near-Infrared (B8) and Red (B4) bands to assess biomass density.

    Args:
        image (ee.Image): The input Sentinel-2 median composite.

    Returns:
        ee.Image: The input image with an additional 'NDVI' band.
    """
    # Select the required spectral bands for NDVI
    nir = image.select('B8') # Near-Infrared
    red = image.select('B4') # Red

    # Apply the NDVI formula: (NIR - Red) / (NIR + Red)
    ndvi = nir.subtract(red).divide(nir.add(red)).rename('NDVI')

    return image.addBands(ndvi)

### 5.2.3. Ecological Classification and Morphology

In [ ]:
def classify_vegetation_health(ndvi_image: ee.Image) -> ee.Image:
    """
    Categorizes NDVI values into three distinct ecological classes based on
    defined thresholds for sclerophyll forests.

    Classes:
    0: Non-Forest / Severe Burn (NDVI <= 0.3) - Bare soil, rock, or charcoal.
    1: Degraded / Regenerating (0.3 < NDVI <= 0.6) - Sparse or young regrowth.
    2: Healthy Forest Canopy (NDVI > 0.6) - Dense, active photosynthesis.

    Args:
        ndvi_image (ee.Image): Single-band NDVI image.

    Returns:
        ee.Image: Categorical image with values 0, 1, and 2.
    """
    # Logic:
    # .gt(threshold) returns 1 if true, 0 otherwise.
    # Adding these results creates the 0, 1, 2 scale.
    is_above_degraded = ndvi_image.gt(THRESHOLD_DEGRADED) # > 0.3
    is_above_healthy  = ndvi_image.gt(THRESHOLD_HEALTHY)  # > 0.6

    # Class 0: both false (0+0)
    # Class 1: only first true (1+0)
    # Class 2: both true (1+1)
    return is_above_degraded.add(is_above_healthy).rename('health_class')


def classify_landforms_tpi(dem_id: str, study_area_geom: ee.Geometry) -> tuple[ee.Image, ee.Image]:
    """
    Uses Topographic Position Index (TPI) to classify terrain.
    Corrected band name: 'elevation' for NASADEM.
    """
    print("Processing TPI analysis (using 'elevation' band)...")

    # Load NASADEM and select the correct band
    dem = ee.Image(dem_id).select('elevation').clip(study_area_geom)

    # Calculate local mean
    local_mean = dem.reduceNeighborhood(
        reducer=ee.Reducer.mean(),
        kernel=ee.Kernel.circle(300, 'meters')
    )

    # TPI = Elevation - Local Mean
    tpi = dem.subtract(local_mean).rename('TPI')

    # Thresholds: valleys < -15m, ridges > 15m
    valleys = tpi.lt(-15).rename('valleys')
    ridges = tpi.gt(15).rename('ridges')

    return valleys, ridges

### 5.2.4. Quantitative Analytics

In [ ]:
def calculate_categorical_areas(
    class_image: ee.Image,
    geometry: ee.Geometry,
    scale: int,
    max_pixels: float
) -> dict:
    """
    Calculates the surface area in hectares for each ecological class
    within the specified study area.

    Args:
        class_image (ee.Image): The classified NDVI image (0, 1, 2).
        geometry (ee.Geometry): The area to perform the calculation over.
        scale (int): Spatial resolution for the calculation.
        max_pixels (float): Maximum limit for pixel processing.

    Returns:
        dict: A dictionary where keys are class IDs and values are hectares.
    """
    # Create an image where each pixel value is its own area in square meters
    # then group by the health class band
    area_image = ee.Image.pixelArea().addBands(class_image)

    # Perform regional reduction to sum up areas per class
    stats = area_image.reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName='class'),
        geometry=geometry,
        scale=scale,
        maxPixels=max_pixels
    ).get('groups')

    # Convert GEE server-side result to a local dictionary
    groups = ee.List(stats).getInfo()

    # Initialize results with 0 to handle cases where a class might be missing
    results = {0: 0.0, 1: 0.0, 2: 0.0}

    for g in groups:
        # Convert square meters to hectares (1 ha = 10,000 m2)
        results[g['class']] = g['sum'] / 10000

    return results


def get_stats_for_mask(mask, mask_name):
    """Calculates hectares for Baseline, Impact, and Recovery within a specific mask."""
    print(f"Calculating multi-temporal stats for: {mask_name}...")

    # Apply the topographic mask to each ecological phase
    baseline_m = pre_fire_class.updateMask(mask)
    impact_m   = impact_class.updateMask(mask)
    recovery_m = recovery_class.updateMask(mask)

    # Calculate hectares using your existing categorical area function
    s_baseline = calculate_categorical_areas(baseline_m, study_area_geom, SCALE_STATS, MAX_PIXELS)
    s_impact   = calculate_categorical_areas(impact_m, study_area_geom, SCALE_STATS, MAX_PIXELS)
    s_recovery = calculate_categorical_areas(recovery_m, study_area_geom, SCALE_STATS, MAX_PIXELS)

    return s_baseline, s_impact, s_recovery

## 5.3. Visualization

### 5.3.1. Interactive Exploration Dashboards

In [ ]:
def visualize_study_area(study_area_features: ee.FeatureCollection, park_names: list[str]):
    """
    Renders the study area boundaries on an interactive map.
    """
    Map = geemap.Map(basemap='HYBRID')
    palette = ['#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#f58231', '#911eb4', '#46f0f0']

    Map.centerObject(study_area_features.geometry(), zoom=9)
    for i, name in enumerate(park_names):
        park_feature = study_area_features.filter(ee.Filter.eq('NAME', name))
        if park_feature.size().getInfo() > 0:
            Map.addLayer(park_feature, {'color': palette[i % len(palette)]}, f"Area: {name}")

    display(Map)


def visualize_ndvi_split_comparison(
    left_image: ee.Image,
    right_image: ee.Image,
    study_area_geom: ee.Geometry,
    left_label: str = 'Before',
    right_label: str = 'After'
) -> None:
    """
    Creates an interactive split-panel map with vertically aligned labels at the bottom.
    """

    # 1. High-contrast NDVI visualization parameters
    ndvi_vis_params = {
        'min': 0,
        'max': 0.8,
        'palette': ['#d73027', '#f46d43', '#fdae61', '#fee08b', '#d9ef8b', '#a6d96a', '#66bd63', '#1a9850']
    }

    # 2. Create Tile Layers
    left_tile = geemap.ee_tile_layer(left_image, ndvi_vis_params, left_label)
    right_tile = geemap.ee_tile_layer(right_image, ndvi_vis_params, right_label)

    # 3. Initialize Map
    Map = geemap.Map(basemap='HYBRID')
    Map.centerObject(study_area_geom, 10)

    # 4. Split Map
    Map.split_map(left_layer=left_tile, right_layer=right_tile)

    # 5. SYMMETRICAL LABELS AT THE BOTTOM
    # Using bottomleft and bottomright ensures they are aligned because
    # there are no bulky default controls there.
    Map.add_text(f" {left_label} ", position='bottomleft')
    Map.add_text(f" {right_label} ", position='bottomright')

    # 6. COLORBAR AT THE TOP-RIGHT
    # Moving the colorbar to the top-right prevents it from overlapping with the labels
    Map.add_colorbar(ndvi_vis_params, label="NDVI Index", position='topright')

    # Add study area boundary
    Map.addLayer(study_area_geom, {'color': 'white', 'fillColor': '00000000', 'width': 2}, 'Study Area Outline')

    display(Map)


def visualize_multitemporal_ndvi(
    pre_ndvi: ee.Image,
    impact_ndvi: ee.Image,
    recovery_ndvi: ee.Image,
    study_area_features: ee.FeatureCollection, # Używamy features zamiast geom dla lepszego stylu
    pre_label: str,
    impact_label: str,
    recovery_label: str
) -> None:
    """
    Renders a multi-temporal interactive map with NDVI layers and a clean vector outline.
    """
    # 1. Initialize map
    Map = geemap.Map(basemap='HYBRID')
    # Use the geometry of the features to center
    Map.centerObject(study_area_features.geometry(), 10)

    # 2. Consistent NDVI visualization palette
    ndvi_vis = {
        'min': 0,
        'max': 0.8,
        'palette': ['#d73027', '#f46d43', '#fdae61', '#fee08b', '#d9ef8b', '#a6d96a', '#66bd63', '#1a9850']
    }

    # 3. Add NDVI layers
    Map.addLayer(pre_ndvi, ndvi_vis, f"1. Baseline: {pre_label}")
    Map.addLayer(impact_ndvi, ndvi_vis, f"2. Fire Impact: {impact_label}")
    Map.addLayer(recovery_ndvi, ndvi_vis, f"3. Recovery: {recovery_label}")

    # 4. Add Colorbar
    Map.add_colorbar(ndvi_vis, label="NDVI Index (Biomass Density)")

    # 5. CLEAN VECTOR OUTLINE (No fill, just a line)
    # We use the .style() method which is the most robust way to get a clean border
    boundary_outline = study_area_features.style(
        color='white',
        fillColor='00000000',
        width=2
    )
    Map.addLayer(boundary_outline, {}, 'Study Area Boundary')

    display(Map)


def visualize_ecological_classification(
    pre_fire_class: ee.Image,
    impact_class: ee.Image,
    recovery_class: ee.Image,
    study_area_features: ee.FeatureCollection,
    threshold_healthy: float,
    threshold_degraded: float
) -> None:
    """
    Renders an interactive map with multi-temporal ecological classification layers
    and a dynamic legend based on the specified NDVI thresholds.

    Args:
        pre_fire_class (ee.Image): Categorical image (0,1,2) for the baseline period.
        impact_class (ee.Image): Categorical image (0,1,2) for the fire impact period.
        recovery_class (ee.Image): Categorical image (0,1,2) for the recovery period.
        study_area_features (ee.FeatureCollection): Vector boundaries for the parks.
        threshold_healthy (float): NDVI threshold used for the 'Healthy' class.
        threshold_degraded (float): NDVI threshold used for the 'Burnt' class.
    """

    # 1. Initialize the interactive map with a stable hybrid background
    Map = geemap.Map(basemap='HYBRID')
    Map.centerObject(study_area_features.geometry(), 9)

    # 2. Define categorical visualization parameters
    # 0: Burnt (Red), 1: Degraded/Regen (Yellow), 2: Healthy (Green)
    class_vis_params = {
        'min': 0,
        'max': 2,
        'palette': ['#d73027', '#fee08b', '#1a9850']
    }

    # 3. Add categorical layers to the map
    Map.addLayer(pre_fire_class, class_vis_params, '1. Baseline State (Classified)')
    Map.addLayer(impact_class, class_vis_params, '2. Fire Impact (Classified)')
    Map.addLayer(recovery_class, class_vis_params, '3. Current Recovery (Classified)')

    # 4. Construct a dynamic legend based on the input thresholds
    legend_keys = [
        f'Healthy Forest (NDVI > {threshold_healthy})',
        f'Degraded / Regenerating ({threshold_degraded} - {threshold_healthy})',
        f'Burnt / Non-forest (NDVI < {threshold_degraded})'
    ]
    legend_colors = ['#1a9850', '#fee08b', '#d73027']

    Map.add_legend(
        keys=legend_keys,
        colors=legend_colors,
        title="Vegetation Health Classes"
    )

    # 5. Add professional white boundary outlines (no fill)
    boundary_style = study_area_features.style(color='white', fillColor='00000000', width=2)
    Map.addLayer(boundary_style, {}, 'Study Area Boundaries')

    # Render the final interactive component
    display(Map)

### 5.3.2. Technical and Validation Layers

In [ ]:
def visualize_topographic_masks(
    valley_mask: ee.Image,
    ridge_mask: ee.Image,
    study_area_features: ee.FeatureCollection
) -> None:
    """
    Renders an interactive map showing the topographic classification (valleys vs ridges)
    overlaid with the study area boundaries.
    """
    # 1. Initialize map with hybrid satellite imagery
    Map_TPI = geemap.Map(basemap='HYBRID')

    # 2. Automatically center the map on the study area
    Map_TPI.centerObject(study_area_features.geometry(), 9)

    # 3. Add Topographic Layers
    # Blue for deep canyons (refugia)
    Map_TPI.addLayer(valley_mask.selfMask(), {'palette': ['#0000FF']}, 'Canyon Refugia (Valleys)')

    # Red for exposed ridges (fire-prone)
    Map_TPI.addLayer(ridge_mask.selfMask(), {'palette': ['#FF0000']}, 'Exposed Ridges')

    # 4. Add Vector Boundaries (Clean white outline)
    boundary_outline = study_area_features.style(color='white', fillColor='00000000', width=2)
    Map_TPI.addLayer(boundary_outline, {}, 'National Park Boundaries')

    # 5. Add legend for clarity
    legend_keys = ['Canyon Refugia (Valleys)', 'Exposed Ridges']
    legend_colors = ['#0000FF', '#FF0000']
    Map_TPI.add_legend(keys=legend_keys, colors=legend_colors, title="Topographic Classes")

    display(Map_TPI)


def visualize_official_fire_history(
    fire_collection_id: str,
    study_area_geom: ee.Geometry,
    study_area_features: ee.FeatureCollection,
    start_date: str = '2019-11-01',
    end_date: str = '2020-05-01'
) -> None:
    """
    Renders the official MODIS Burned Area product for the 2019/20 'Black Summer' season
    overlaid with study area boundaries for validation purposes.
    """
    print(f"Loading official MODIS Burned Area data from {start_date} to {end_date}...")

    # 1. Load and filter the MODIS Burned Area collection
    fire_data = ee.ImageCollection(fire_collection_id) \
        .filterDate(start_date, end_date) \
        .select('BurnDate') \
        .max() \
        .clip(study_area_geom)

    # 2. Initialize the map
    Map_Fire = geemap.Map(basemap='HYBRID')
    Map_Fire.centerObject(study_area_geom, 9)

    # 3. Add the Official Burn Scar
    fire_vis = {'min': 1, 'max': 366, 'palette': ['#FF4500']}
    Map_Fire.addLayer(fire_data.gt(0), fire_vis, 'Official MODIS Burn Scar (2019/20)')

    # 4. Add Vector Boundaries
    boundary_outline = study_area_features.style(color='white', fillColor='00000000', width=2)
    Map_Fire.addLayer(boundary_outline, {}, 'National Park Boundaries')

    # 5. Add Legend - FIXED: changed 'labels' to 'keys'
    Map_Fire.add_legend(
        title="Fire Validation",
        keys=["Official Burned Area (MODIS)"],
        colors=["#FF4500"]
    )

    display(Map_Fire)

# 6. Results and Interpretation

## 6.1. Initialization the Earth Engine session

In [ ]:
authenticate_project(PROJECT_ID)

## 6.2. Spatial Extent: The Regional National Park Cluster

This section establishes the spatial and administrative foundation of the study area. Located in New South Wales, Australia, the analysis focuses on a regional cluster comprising the Blue Mountains, Wollemi, and Yengo National Parks. Together, these protected areas form a continuous wilderness corridor of nearly one million hectares (~953,000 ha), a figure verified using the Earth Engine spherical geometry engine. As a core part of the UNESCO Greater Blue Mountains World Heritage Area, the metadata—sourced from the World Database on Protected Areas (WDPA)—highlights the region's global ecological significance. The cluster is characterized by a high concentration of strict protection levels, specifically IUCN Categories Ia (Strict Nature Reserve), Ib (Wilderness Area), and II (National Park), ensuring the preservation of large-scale natural processes and biodiversity.

In [ ]:
# Fetching official park boundaries and merged geometry from WDPA
study_area_geom, study_area_features = fetch_study_area(
    park_names=PARK_NAMES,
    iso_code=COUNTRY_ISO,
    collection_id=WDPA_PATH
)

In [ ]:
# Calculating study area in hectres
actual_area_ha = study_area_geom.area().divide(10000).getInfo()
print(f"Verified Study Area Size: {actual_area_ha:,.0f} hectares")

In [ ]:
# Extracting and inspecting administrative metadata: IUCN status and reported park areas
gdf_parks = analyze_vector_metadata(study_area_features)

In [ ]:
# Study area visualization
visualize_study_area(
    study_area_features=study_area_features,
    park_names=PARK_NAMES
)

## 6.3. Generating cloud-masked median composites for the three ecological phases

In [ ]:
# Individual satellite images often contain "noise" like clouds, shadows,
# or smoke, which distort NDVI values. The goal of this code is to create
# composites to obtain a stable, representative "snapshot" of the forest
# for each ecological phase.

# The function filters the entire Sentinel-2 collection for each
# date range and applies a "Median Reducer." For every 10m pixel, it looks at
# all available observations (e.g., 70+ images per summer) and selects the
# middle value. This statistically ignores outliers (bright clouds/dark shadows),
# resulting in a clean, seamless mosaic ready for comparative analysis.

img_pre, img_fire, img_rec = generate_fire_analysis_composites(
    study_area_geom=study_area_geom,
    s2_collection_id=S2_PATH,
    cloud_threshold=MAX_CLOUD_PERCENTAGE,
    pre_start=PRE_FIRE_START, pre_end=PRE_FIRE_END,
    fire_start=FIRE_EVENT_START, fire_end=FIRE_EVENT_END,
    rec_start=POST_FIRE_START, rec_end=POST_FIRE_END
)

## 6.4. Applying custom function and isolate the NDVI band

In [ ]:
pre_fire_ndvi = calculate_ndvi(img_pre).select('NDVI')
impact_ndvi   = calculate_ndvi(img_fire).select('NDVI')
recovery_ndvi = calculate_ndvi(img_rec).select('NDVI')

## 6.5. Visualizing the Disturbance: NDVI Split-Panel Comparison

 This interactive split-panel map provides a direct comparison between the forest's baseline state (2016/17) and the peak fire impact (2020). By sliding the divider, one can observe the catastrophic shift in the NDVI signal. The vibrant green areas (high NDVI), representing a continuous healthy canopy, are replaced by massive russet and red zones (low NDVI). This visual "charcoal footprint" illustrates how the 'Black Summer' fires effectively eliminated the photosynthetic capacity of the forest across almost the entire study area.

In [ ]:
visualize_ndvi_split_comparison(
    left_image=pre_fire_ndvi,
    right_image=impact_ndvi,
    study_area_geom=study_area_geom,
    left_label=f'{PRE_FIRE_START}  -  {PRE_FIRE_END}',
    right_label=f'{FIRE_EVENT_START}  -  {FIRE_EVENT_END}',
)

## 6.6. Visualization Post-fire Recovery Trajectory

This comparison highlights the six-year regeneration phase (2020 to 2026). By shifting the slider from the charred landscape of the 2020 "impact" to the current state, we observe a massive return of the NDVI signal. This rapid "green-up" is characteristic of Australian eucalyptus forests, which utilize epicormic sprouting to push new leaves directly from burnt trunks. While the map shows a widespread return of vegetation, this stage represents an early successional phase where dense, young regrowth often results in NDVI values even higher than the original old-growth baseline.

In [ ]:
visualize_ndvi_split_comparison(
    left_image=impact_ndvi,
    right_image=recovery_ndvi,
    study_area_geom=study_area_geom,
    left_label=f'{FIRE_EVENT_START}  -  {FIRE_EVENT_END}',
    right_label=f'{POST_FIRE_END}  -  {POST_FIRE_END}',
)

## 6.7. Integrated Temporal Explorer: Tracking NDVI Trajectories (2016–2026)

This multi-temporal interactive map consolidates the entire study timeline into a single interface. Using the layer control in the top-right corner, users can toggle between the 2016 Baseline, the 2020 Fire Impact, and the 2026 Recovery phases. This view is essential for observing the landscape-level "pulse" of the forest: from its stable pre-fire state, through the near-total elimination of green biomass during the 'Black Summer', to the vigorous but structurally different regrowth observed today.

In [ ]:
visualize_multitemporal_ndvi(
    pre_ndvi=pre_fire_ndvi,
    impact_ndvi=impact_ndvi,
    recovery_ndvi=recovery_ndvi,
    study_area_features=study_area_features,
    pre_label=f"{PRE_FIRE_START} to {PRE_FIRE_END}",
    impact_label=f"{FIRE_EVENT_START} to {FIRE_EVENT_END}",
    recovery_label=f"{POST_FIRE_START} to {POST_FIRE_END}"
)

## 6.8. Converting continuous NDVI into discrete health classes (Burnt, Degraded, Healthy)

In [ ]:
# This categorization is based on NDVI thresholding
# - NDVI < THRESHOLD_DEGRADED (0.3): Represents non-photosynthetic surfaces (bare soil, rock, or charcoal).
# - NDVI > THRESHOLD_HEALTHY (0.6): Represents the baseline for healthy, high-biomass forest canopy.
pre_fire_class = classify_vegetation_health(pre_fire_ndvi)
impact_class   = classify_vegetation_health(impact_ndvi)
recovery_class = classify_vegetation_health(recovery_ndvi)

This visualization reveals a recovery paradox within the study area. While the 2020 map shows a near-total collapse of the ecosystem (dominated by red pixels), the 2025-2026 map shows an explosion of green that exceeds the 2016 baseline. To quantify this, the following NDVI thresholds were applied:

  - Burnt / Non-forest (NDVI < 0.3): Represents areas where fire completely removed the canopy, exposing bare soil, rock, or charcoal. In these zones, the spectral signature is dominated by the ground surface rather than active photosynthesis.

  - Degraded / Regenerating (0.3 - 0.6): Captures the transitional state—either sparse forests that survived with damage, or early-stage regrowth where the vegetation is present but the canopy has not yet reached full maturity or density.

  - Healthy Forest (NDVI > 0.6): Indicates a dense, robust photosynthetic canopy.

The intense greening in 2025-2026 is driven by a massive "green-up" effect: rapid growth of chlorophyll-rich pioneer species and dense epicormic sprouting. Fuelled by recent high rainfall, these young leaves produce a stronger spectral signal than the mature, stable forest of 2016, pushing vast areas from the "Burnt"/"Degraded" category directly into the "Healthy" category within just six years.

In [ ]:
visualize_ecological_classification(
    pre_fire_class=pre_fire_class,
    impact_class=impact_class,
    recovery_class=recovery_class,
    study_area_features=study_area_features,
    threshold_healthy=THRESHOLD_HEALTHY,
    threshold_degraded=THRESHOLD_DEGRADED
)

## 6.9. Calculating total area in hectares for each class across the whole cluster

In [ ]:
# Calculate the total surface area in hectares for each ecological category.
# Each phase (Baseline, Impact, Recovery) is now categorized into
# three Ecological Categories: Burnt (0), Degraded (1), and Healthy (2).
stats_baseline = calculate_categorical_areas(pre_fire_class, study_area_geom, SCALE_STATS, MAX_PIXELS)
stats_impact   = calculate_categorical_areas(impact_class, study_area_geom, SCALE_STATS, MAX_PIXELS)
stats_recovery = calculate_categorical_areas(recovery_class, study_area_geom, SCALE_STATS, MAX_PIXELS)

## 6.10. Building main analytical table

In [ ]:
df_area_report = pd.DataFrame({
    'Ecological Category': [
        'Burnt / Non-forest (NDVI < 0.3)',
        'Degraded / Regenerating (0.3 - 0.6)',
        'Healthy Forest (NDVI > 0.6)'
    ],
    'Baseline [ha]': [stats_baseline[0], stats_baseline[1], stats_baseline[2]],
    'Fire Impact [ha]': [stats_impact[0], stats_impact[1], stats_impact[2]],
    'Current Recovery [ha]': [stats_recovery[0], stats_recovery[1], stats_recovery[2]]
})

## 6.11. Quantitative Insights: Impact Severity and Recovery Metrics



This section translates the raw hectare data into performance indicators that quantify the disaster's magnitude and the ecosystem's response. These metrics are strictly tied to our classification logic, where NDVI > 0.6 is the benchmark for a Healthy Forest (dense, high-biomass canopy).

The calculation follows two directions:

  - Impact Severity (98.7% Canopy Loss): This is the most critical ecological finding. It indicates that nearly 99% of the pre-existing high-biomass forest (NDVI > 0.6) was degraded or destroyed during the crisis. This does not mean the trees were all physically removed, but rather that their photosynthetic "green" signal was almost entirely eliminated, indicating severe canopy scorching or total incineration.

  - Spatial Scale (47.5% Burn Scar): On a regional level, nearly half of the entire 951,445-hectare cluster was converted into a "severe burn scar" (NDVI < 0.3). This represents a geographic footprint of over 450,000 hectares where the satellite sensor detected only charred remains, bare soil, or rock, with no significant living vegetation remaining.

In [ ]:
total_area_ha = stats_baseline[0] + stats_baseline[1] + stats_baseline[2]
loss_of_existing_forest_pct = ((stats_baseline[2] - stats_impact[2]) / stats_baseline[2]) * 100
park_burned_share_pct = (stats_impact[0] / total_area_ha) * 100

In [ ]:
print(f"DISTURBANCE SUMMARY: THE IMPACT OF 'BLACK SUMMER'")
print(f"Total Analyzed Landscape: {total_area_ha:,.0f} hectares")

In [ ]:
print(f"-> IMPACT SEVERITY: {loss_of_existing_forest_pct:.1f}% loss of pre-existing healthy forest.")
print(f"-> SPATIAL SCALE:    {park_burned_share_pct:.1f}% of the cluster area became a burn scar.")

In [ ]:
display(df_area_report[['Ecological Category', 'Baseline [ha]', 'Fire Impact [ha]', 'Current Recovery [ha]']])

The regional statistics for National Park cluster reveal a narrative of extreme ecosystem collapse followed by an aggressive, yet potentially misleading, spectral recovery.

  - Extent of Destruction (2020): The findings document a near-total functional collapse of the forest at the peak of the 'Black Summer' crisis. The loss of 98.7% of the "Healthy Forest" area (plummeting from 246,047 ha to just 3,113 ha) is a staggering indicator of fire severity. This confirms that the disturbance was not a typical seasonal event but a catastrophic crown fire that eliminated the photosynthetic canopy across nearly the entire regional cluster.

  - Regeneration Dynamics and the "Hyper-green" Observation: By 2026, the data shows a massive statistical surge—an 11,524% increase in the "Healthy Forest" category relative to the 2020 low. Remarkably, the current healthy area (361,899 ha) significantly exceeds the 2016 baseline. This "hyper-green" state represents a classic post-fire green-up paradox, driven by chlorophyll-dense epicormic shoots and pioneer species.

  - The Analytical Trap: While these regional numbers suggest a "super-recovery," they represent a significant analytical trap—an observation already hinted at by the deceptively uniform greenness seen in the earlier NDVI and classification maps. The fire-adapted eucalyptus forests dominate the regional spectral signal and recover so vigorously that they mask the fate of more fragile ecosystems. To understand the true ecological impact, it is critical to move beyond regional averages. Further analysis must be specifically targeted at the high-value rainforest zones. This requires incorporating topography into our model: by using Topographic Position Index (TPI), we adopt the ecological assumption that deep, moist canyons serve as the primary refugia for fire-sensitive rainforest species. Without isolating these Canyon Refugia from the Exposed Ridges, we cannot determine if the most vulnerable flora has actually recovered or if it has been permanently replaced by opportunistic pioneer scrub.

## 6.12. Identifying Canyon Refugia and Exposed Ridges using TPI analysis

In [ ]:
# This function implements the Topographic Position Index (TPI) algorithm to
# segment the landscape into two critical ecological zones:
#
# 1. Canyon Refugia (valley_mask): Identified as pixels significantly lower
#     than their surroundings (TPI < -15m). These represent the moist,
#     sheltered canyon floors where fire-sensitive rainforests reside.
#
# 2. Exposed Ridges (ridge_mask): Identified as pixels significantly higher
#     than their surroundings (TPI > 15m). These are the dry, wind-exposed
#     plateaus and ridgetops dominated by fire-adapted eucalyptus.
#
# Logic: The algorithm calculates the mean elevation within a 300m radius of
# each pixel and subtracts it from the pixel's actual elevation. This relative
# measure allows for the detection of canyons and peaks regardless of their
# absolute height above sea level.
valley_mask, ridge_mask = classify_landforms_tpi(DEM_PATH, study_area_geom)

## 6.13. Topographic Segmentation: Delineating Canyon Refugia and Exposed Ridges

This interactive map visualizes the results of the Topographic Position Index (TPI) analysis used to segment the landscape into two critical zones: Canyon Refugia (Blue) and Exposed Ridges (Red). By comparing pixel elevation to the local neighborhood mean, we identified the moist, protected valley floors—critical habitats for fire-sensitive species like the Wollemi Pine.

  - *Technical Note: Because these topographic features are naturally narrow—canyons often span only 100 to 300 meters—they are not easily visible at a regional scale. Users must zoom in significantly to see the intricate network of "blue threads" representing the valley floors.*

In [ ]:
visualize_topographic_masks(valley_mask, ridge_mask, study_area_features)

## 6.14. Calculating stats specifically for Valleys and Ridges

In [ ]:
# These calls execute sub-regional statistics by spatially intersecting the
# topographic masks with the multi-temporal vegetation health classes.
#
# his step isolates the hectares for each ecological category within
# specific terrain niches. By running these stats separately for "Canyon Refugia"
# and "Exposed Ridges," we generate the evidence needed to compare the fire
# behavior in fire-sensitive vs. fire-adapted zones across all three periods:
# Baseline (2016), Fire Impact (2020), and Recovery (2026).
v_baseline, v_impact, v_recovery = get_stats_for_mask(valley_mask, "Canyon Refugia")
r_baseline, r_impact, r_recovery = get_stats_for_mask(ridge_mask, "Exposed Ridges")

## 6.15. Comparing fire behavior between topographic niches


In [ ]:
df_comparison = pd.DataFrame({
    'Ecological Category': ['Burnt / Non-forest', 'Degraded / Regen', 'Healthy Forest'],
    'Valleys 2016 [ha]': [v_baseline[0], v_baseline[1], v_baseline[2]],
    'Valleys 2020 [ha]': [v_impact[0], v_impact[1], v_impact[2]],
    'Valleys 2026 [ha]': [v_recovery[0], v_recovery[1], v_recovery[2]],
    'Ridges 2016 [ha]':  [r_baseline[0], r_baseline[1], r_baseline[2]],
    'Ridges 2020 [ha]':  [r_impact[0], r_impact[1], r_impact[2]],
    'Ridges 2026 [ha]':  [r_recovery[0], r_recovery[1], r_recovery[2]]
})

display(df_comparison)

The comparative analysis between Canyon Refugia (Valleys) and Exposed Ridges provides the most significant ecological insight of this study. It reveals that the 2019/20 fires bypassed the natural protective mechanisms of the landscape.

  - The Collapse of Canyon Refugia (2016 vs. 2020):
  Historically, the deep sandstone canyons have acted as fire-proof sanctuaries. In 2016, these valleys were remarkably stable, supporting over 90,348 ha of Healthy Forest. By 2020, this area plummeted to a mere 1,725 ha—a catastrophic 98.1% loss. The fact that the "Burnt" category in valleys surged from 9k ha to nearly 79k ha proves that the unprecedented drought lowered fuel moisture to a point where the fire successfully breached the moist microclimates of the canyon floors, likely devastating fire-sensitive rainforest species and Araucariaceae (Wollemi Pine) habitats.

  - Ridge-top Extremes:
  On the exposed ridges, the destruction was even more absolute, with healthy forest area dropping from 43,985 ha to only 284 ha (a 99.3% reduction). This confirms that the ridges acted as high-energy "fire highways," where near-total incineration occurred due to exposure to wind and extreme heat.

  - The 2026 Recovery and the Green-up Signal:
  By 2026, the "Healthy Forest" category in both niches has exceeded the 2016 baseline (117,676 ha in Valleys and 72,580 ha on Ridges). This rapid spectral recovery is a testament to the intense successional drive of the Australian bush. However, the fact that the 2026 values are significantly higher than the 2016 baseline confirms that we are observing an early-successional "green-up" phase. The current canopy is likely composed of high-chlorophyll pioneer species and dense epicormic regrowth, which produces a stronger NDVI signal than the complex, multi-story old-growth forest that existed before the fires.



## 6.16. Independent Validation: Correlating NDVI Loss with Official MODIS Fire Data

This final visualization overlays the official MODIS Burned Area (MCD64A1) product for the 2019/20 season onto the study area. The orange-red mask represents pixels where thermal and spectral sensors detected active fire or charcoal residues during the 'Black Summer' period.

  - Observations: There is a very high spatial correlation between the official MODIS fire scars and the drastic NDVI drops identified in our Sentinel-2 classification (Impact 2020). The fact that our "Burnt" category (<0.3 NDVI) aligns almost perfectly with the independent MODIS data confirms that manually calculated NDVI is a robust and highly reliable proxy for mapping the true extent of forest disturbances. While MODIS provides a global overview at 500m resolution, our Sentinel-2 analysis refines this by showing the same patterns at a much higher 10m precision.

In [ ]:
visualize_official_fire_history(
    fire_collection_id=FIRE_PATH,
    study_area_geom=study_area_geom,
    study_area_features=study_area_features
)

# 7. Conclusion

**Potential and Limitations of NDVI in Post-Fire Monitoring**

This analysis demonstrates that manually calculated NDVI is a robust tool for mapping the magnitude of environmental disasters. The results clearly delineate the massive area of biomass loss in 2020. Our classification shows a strong spatial correlation with the official MODIS Burned Area (MCD64A1) product.

However, evaluating forest recovery solely through NDVI can be highly deceptive. While 2026 data suggest a "healthier" state than the 2016 baseline—even in fire-sensitive rainforests—we know that the full restoration of such complex ecosystems takes decades, not years. The "hyper-green" signal in 2026 is driven by high recent rainfall and the fact that the 2020 mega-fire cleared the landscape of smaller, pre-existing fire scars seen in 2016. For a true ecological assessment, structural (height) data would be required—such as LiDAR from the GEDI mission—to differentiate between dense, low-level pioneer regrowth and the complex, multi-story old-growth forest that existed prior to the crisis.